In [1]:
"""
================================================================================
PROJECT STEP 1: NOMINAL GROUND TRUTH EXTRACTION (STATIC PHASE)
================================================================================
DESCRIPTION:
This module establishes the core mathematical anchor points [A_nominal] and 
[B_nominal] for an open-loop unstable Continuous Stirred Tank Reactor (CSTR). 
We reject low-fidelity finite-difference approximations in favor of an exact 
analytical vector field mapping technique. This modular data pipeline separates
the hardware logging from matrix processing using object-oriented classes.

HARDWARE & EQUIPMENT SPECIFICATIONS:
- Primary Sensor Array: Integrated dual-channel inline sensor tracking a 2D 
  continuous state vector x(t) = [C_A(t), T(t)]^T.
  * Channel 1 (Concentration): High-precision inline spectrophotometer.
    Operational Bounds: [0.0, 2.0] kmol/m^3. Sampling rate: 100 Hz (dt = 0.01s).
  * Channel 2 (Temperature): Industrial Class-A Resistance Temperature Detector (RTD).
    Operational Bounds: [300.0, 500.0] K. Sampling rate: 100 Hz (dt = 0.01s).
- Actuator Assembly (u): Electronically controlled pneumatic coolant jacket valve.
  The input manifold u(t) dictates coolant temperature T_c(t).
  Physical Hardware Bounds: [250.0, 450.0] K.
================================================================================
"""

import numpy as np

class Initialization:
    """
    Class 1: Handles plant parameter storage, physical ODE evaluations, 
    and simulates the real-time sensor streaming phase under input dither excitation.
    """
    def __init__(self):
        # 1. Hardware Sensor Setup
        self.dt = 0.01          # 100 Hz hardware sensor sampling frequency
        self.n_samples = 1000   # Number of continuous snapshots collected
        
        # 2. Plant Physical Ground-Truth Constants
        self.q_V = 1.0          # Volumetric space velocity (q/V) [s^-1]
        self.C_Af = 1.0         # Feed concentration of reactant A [kmol/m^3]
        self.T_f = 350.0        # Feed temperature [K]
        self.k_0 = 1e8          # Arrhenius pre-exponential kinetic constant [s^-1]
        self.E_R = 6000.0       # Activation energy over gas constant (E/R) [K]
        self.dH_term = 2e5      # Dimensionless adiabatic heat of reaction term [K*m^3/kmol]
        self.UA_term = 0.5      # Jacket heat transfer coefficient term [s^-1]
        
        # 3. Unstable Steady-State Target Equilibrium (The Local Origin)
        self.C_A_ss = 0.5       # Steady-state concentration [kmol/m^3]
        self.T_ss = 400.0       # Steady-state temperature [K]
        self.T_c_ss = 350.0     # Steady-state nominal coolant jacket temperature [K]import numpy as np

class Initialization:
    """
    Class 1: Handles plant parameter storage, physical ODE evaluations, 
    and simulates the real-time sensor streaming phase under input dither excitation.
    """
    def __init__(self):
        # 1. Hardware Sensor Setup
        self.dt = 0.01          # 100 Hz hardware sensor sampling frequency
        self.n_samples = 1000   # Number of continuous snapshots collected
        
        # 2. Plant Physical Ground-Truth Constants
        self.q_V = 1.0          # Volumetric space velocity (q/V) [s^-1]
        self.C_Af = 1.0         # Feed concentration of reactant A [kmol/m^3]
        self.T_f = 350.0        # Feed temperature [K]
        self.k_0 = 1e8          # Arrhenius pre-exponential kinetic constant [s^-1]
        self.E_R = 6000.0       # Activation energy over gas constant (E/R) [K]
        self.dH_term = 2e5      # Dimensionless adiabatic heat of reaction term [K*m^3/kmol]
        self.UA_term = 0.5      # Jacket heat transfer coefficient term [s^-1]
        
        # 3. Unstable Steady-State Target Equilibrium (The Local Origin)
        self.C_A_ss = 0.5       # Steady-state concentration [kmol/m^3]
        self.T_ss = 400.0       # Steady-state temperature [K]
        self.T_c_ss = 350.0     # Steady-state nominal coolant jacket temperature [K]

    def cstr_nonlinear_dynamics(self, C_A, T, T_c):
        """
        Evaluates the exact, non-linear physical ordinary differential equations 
        (mass balance and energy balance) governing the internal material and energy 
        balances of the reactor.
        """
        # Arrhenius rate law expression
        reaction_rate = self.k_0 * np.exp(-self.E_R / T) * C_A
        
        # Mass Balance: d(C_A)/dt
        dC_A = self.q_V * (self.C_Af - C_A) - reaction_rate
        
        # Energy Balance: d(T)/dt
        dT = self.q_V * (self.T_f - T) + self.dH_term * reaction_rate - self.UA_term * (T - T_c)
        
        return np.array([dC_A, dT])

    def stream_sensor_data(self):
         """
        Responsible for simulating the open loop system.
        """
    np.random.seed(42)
    
    # 1. Pre-allocate state and control trajectories over time
    C_A = np.zeros(self.n_samples)
    T = np.zeros(self.n_samples)
    T_c = np.zeros(self.n_samples)
    
    X_dot_analytical = np.zeros((2, self.n_samples))
    
    # 2. Set initial values (C_A0, T0)
    C_A[0] = 0.55
    T[0] = 405.0
    
    for k in range(self.n_samples - 1):
        # Compute excitation signal
        T_c[k] = self.T_c_ss + np.sin(k * 0.05) * 8.0 + np.random.normal(0, 0.2)
        
        # Evaluate current time derivatives: x_dot = [dC_A/dt, dT/dt]
        x_dot = self.cstr_nonlinear_dynamics(C_A[k], T[k], T_c[k])
        X_dot_analytical[:, k] = x_dot
        
        # Euler step: Vector update using derivatives
        C_A[k + 1] = C_A[k] + x_dot[0] * self.dt
        T[k + 1]   = T[k]   + x_dot[1] * self.dt
        
    # Final step derivative evaluation
    T_c[-1] = self.T_c_ss + np.sin((self.n_samples - 1) * 0.05) * 8.0 + np.random.normal(0, 0.2)
    X_dot_analytical[:, -1] = self.cstr_nonlinear_dynamics(C_A[-1], T[-1], T_c[-1])
    
    # Construct deviation matrices relative to target equilibrium
    X_deviations = np.vstack([C_A - self.C_A_ss, T - self.T_ss])
    U_deviations = np.vstack([T_c - self.T_c_ss])


class GetGroundTruth:
    """
    Class 2: Consumes the raw data streams from the Initialization pipeline,
    constructs the data-augmented matrix manifolds, and extracts the 
    nominal A and B matrices via linear least-squares regression.
    """
    def __init__(self, init_instance):
        self.plant = init_instance

    def compute_nominal_matrices(self):
        """
        Executes the data-augmented regression pipeline over the analytic 
        vector field snapshots to uncover the flat local linear grid.
        """
        # Fetch the active sensor and derivative streams from Class 1
        X, U, X_dot = self.plant.stream_sensor_data()
        
        # Data Augmentation: Stack State (X) and Control Input (U) into Matrix Omega
        # Dimensions: (3 x N_SAMPLES)
        Omega = np.vstack([X, U])
        
        # Execute Moore-Penrose Pseudoinverse to resolve: X_dot = [A | B] * Omega
        A_B_augmented = X_dot @ np.linalg.pinv(Omega)
        
        # Slice the augmented mapping into independent nominal matrices
        A_nominal = A_B_augmented[:, :2]
        B_nominal = A_B_augmented[:, 2:3]
        
        return A_nominal, B_nominal


# ==============================================================================
# ROUTINE EXECUTION AND STABILITY VERIFICATION
# ==============================================================================
if __name__ == "__main__":
    # 1. Instantiate the classes
    plant_setup = Initialization()
    truth_extractor = GetGroundTruth(plant_setup)
    
    # 2. Compute the exact ground-truth matrices
    A_nominal, B_nominal = truth_extractor.compute_nominal_matrices()
    
    print("====================================================================")
    print("STATIC PHASE COMPLETE: NOMINAL GROUND TRUTH MATRICES EXTRACTION")
    print("====================================================================")
    print("A_nominal:\n", A_nominal)
    print("\nB_nominal:\n", B_nominal)
    
    # 3. Perform eigenvalue decomposition to prove open-loop positive stability
    eigenvalues = np.linalg.eigvals(A_nominal)
    print("\nOpen-Loop Eigenvalues (System Exponents):", eigenvalues)
    print("Verification: Contains positive exponent =", any(eigenvalues > 0))
    print("====================================================================")

IndentationError: unexpected indent (1344222524.py, line 129)